# Argument realization in an Ojibwe corpus: frequency counts

In this notebook, we will attempt to parse the frequency counts for argument realization, splitting on 3 cases:
1. Animacy: Is there a difference in the ratio of overt animate vs. inanimate arguments?
2. SAPs: Is there a difference in the ratio of overt SAP vs. non-SAP arguments?
3. Obviation: Is there a difference in the ratio of overt obviative vs. proximate arguments? (only 3-3 VTAs)

In [1]:
# get the path to the .conllu treebank
from grammar_modules.disambiguation import REPO_ROOT
CORPUS_PATH = REPO_ROOT / "data" / "treebanks" / "Chi_mewinzha_full.conllu"

### Case 1: Animacy

We will start with animacy. First we will define code that can collect the relevant attributes from a `.conllu` formatted treebank, then we will use this to parse animacy counts.

In [31]:
from conllu import parse_incr

animate_nominal = "NA" 
inanimate_nominal = "NI"

animate_subj_marker = {
    "1SgSubj", "InclSubj", "ExclSubj", "2SgSubj", "2PlSubj",
    "3SgProxSubj", "3PlProxSubj", "3SgObvSubj", "3PlObvSubj",
}
animate_obj_marker = {
    "1SgObj", "InclObj", "ExclObj", "2SgObj", "2PlObj",
    "3SgProxObj", "3PlProxObj", "3SgObvObj", "3PlObvObj",
}
inanimate_subj_marker = {"0SgSubj", "0PlSubj", "0SgObvSubj", "0PlObvSubj"}
inanimate_obj_marker = {"0SgObj", "0PlObj", "0SgObvObj", "0PlObvObj"}

animate_subj_count = animate_obj_count = inanimate_subj_count = inanimate_obj_count = 0
animate_subj_verbs_count = animate_obj_verbs_count = inanimate_subj_verbs_count = inanimate_obj_verbs_count = 0

# for each sentence in treebank, iterate over tokens and collect xpos + deprel 
# (FST tags + dependency relation) and append to counts for each given case
with open(CORPUS_PATH, encoding="utf-8") as f:
    for sent in parse_incr(f):
        for tok in sent:
            xpos = tok.get("xpos")
            if not xpos:
                continue

            # nominal counts
            if animate_nominal in xpos and tok["deprel"] == "nsubj":
                animate_subj_count += 1
            if animate_nominal in xpos and tok["deprel"] == "obj":
                animate_obj_count += 1
            if inanimate_nominal in xpos and tok["deprel"] == "nsubj":
                inanimate_subj_count += 1
            if inanimate_nominal in xpos and tok["deprel"] == "obj":
                inanimate_obj_count += 1

            # verbal person/number markers
            if any(m in xpos for m in animate_subj_marker):
                animate_subj_verbs_count += 1
            if any(m in xpos for m in animate_obj_marker):
                animate_obj_verbs_count += 1
            if any(m in xpos for m in inanimate_subj_marker):
                inanimate_subj_verbs_count += 1
            if any(m in xpos for m in inanimate_obj_marker):
                inanimate_obj_verbs_count += 1

print(
    f"Animate subjects (nominals): {animate_subj_count}",
    f"Animate objects (nominals): {animate_obj_count}",
    f"Inanimate subjects (nominals): {inanimate_subj_count}",
    f"Inanimate objects (nominals): {inanimate_obj_count}",
    sep="\n",
)

print(
    f"Total animate subjects (verbs): {animate_subj_verbs_count}",
    f"Total animate objects (verbs):  {animate_obj_verbs_count}",
    f"Total inanimate subjects (verbs): {inanimate_subj_verbs_count}",
    f"Total inanimate objects (verbs): {inanimate_obj_verbs_count}",
    sep="\n",
)


Animate subjects (nominals): 165
Animate objects (nominals): 106
Inanimate subjects (nominals): 44
Inanimate objects (nominals): 140
Total animate subjects (verbs): 728
Total animate objects (verbs):  187
Total inanimate subjects (verbs): 75
Total inanimate objects (verbs): 152


In [32]:
# create stats tables
import pandas as pd

stats = [
    {"argument type": "animate subject", "total": animate_subj_verbs_count, "overt": animate_subj_count,},
    {"argument type": "animate object", "total": animate_obj_verbs_count,  "overt": animate_obj_count,},
    {"argument type": "inanimate subject", "total": inanimate_subj_verbs_count, "overt": inanimate_subj_count,},
    {"argument type": "inanimate object", "total": inanimate_obj_verbs_count,  "overt": inanimate_obj_count,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

    argument type  total  overt  covert  overt_ratio
  animate subject    728    165     563     0.226648
   animate object    187    106      81     0.566845
inanimate subject     75     44      31     0.586667
 inanimate object    152    140      12     0.921053


### Case 2: SAPs

Next, we will parse overt SAPs vs non-SAPs. The prediction is that SAPs have a very low rate of overt arguments compared to non-SAPs.

In [34]:
from conllu import parse_incr

personal_pronoun = "PRONPer"
SAP_argument = {"1Sg", "Incl", "Excl", "2Sg", "2Pl"}

SAP_subj_marker = {
    "1SgSubj", "InclSubj", "ExclSubj", "2SgSubj", "2PlSubj",
}
SAP_obj_marker = {
    "1SgObj", "InclObj", "ExclObj", "2SgObj", "2PlObj",
}
non_SAP_subj_marker = {
    "3SgProxSubj", "3PlProxSubj", "3SgObvSubj", "3PlObvSubj", 
    "0SgSubj", "0PlSubj", "0SgObvSubj", "0PlObvSubj"
    }
non_SAP_obj_marker = {
    "3SgProxObj", "3PlProxObj", "3SgObvObj", "3PlObvObj",
    "0SgObj", "0PlObj", "0SgObvObj", "0PlObvObj",
    }


SAP_subj_count = SAP_obj_count = non_SAP_subj_count = non_SAP_obj_count = 0
SAP_subj_verbs_count = SAP_obj_verbs_count = non_SAP_subj_verbs_count = non_SAP_obj_verbs_count = 0

with open(CORPUS_PATH, encoding="utf-8") as f:
    for sent in parse_incr(f):
        for tok in sent:
            xpos = tok.get("xpos")
            if not xpos:
                continue

            if personal_pronoun in xpos and any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "nsubj":
                SAP_subj_count += 1
            if personal_pronoun in xpos and any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "obj":
                SAP_obj_count += 1
            if not any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "nsubj":
                non_SAP_subj_count += 1
            if not any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "obj":
                non_SAP_obj_count += 1

            if any(m in xpos for m in SAP_subj_marker):
                SAP_subj_verbs_count += 1
            if any(m in xpos for m in SAP_obj_marker):
                SAP_obj_verbs_count += 1
            if any(m in xpos for m in non_SAP_subj_marker):
                non_SAP_subj_verbs_count += 1
            if any(m in xpos for m in non_SAP_obj_marker):
                non_SAP_obj_verbs_count += 1


print(
    "---OVERT SAP/non-SAP arguments---",
    f"SAP subjects: {SAP_subj_count}",
    f"SAP objects: {SAP_obj_count}",
    f"non-SAP subjects: {non_SAP_subj_count}",
    f"non-SAP objects: {non_SAP_obj_count}",
    sep="\n",
)

print(
    "---TOTAL SAP/non-SAP arguments---",
    f"SAP subjects: {SAP_subj_verbs_count}",
    f"SAP objects: {SAP_obj_verbs_count}",
    f"non-SAP subjects: {non_SAP_subj_verbs_count}",
    f"non-SAP objects: {non_SAP_obj_verbs_count}",
    sep="\n",
)


# TODO: I get 76 total arguments here, but 74 in the word_order.ipynb... 
#       need to figure out why the discrepancy. probably the logic doesn't match up 100% somewhere

---OVERT SAP/non-SAP arguments---
SAP subjects: 20
SAP objects: 4
non-SAP subjects: 157
non-SAP objects: 230
---TOTAL SAP/non-SAP arguments---
SAP subjects: 259
SAP objects: 39
non-SAP subjects: 544
non-SAP objects: 300


In [36]:
# create stats tables
import pandas as pd

stats = [
    {"argument type": "SAP subjects", "total": SAP_subj_verbs_count, "overt": SAP_subj_count,},
    {"argument type": "SAP objects", "total": SAP_obj_verbs_count,  "overt": SAP_obj_count,},
    {"argument type": "non-SAP subjects", "total": non_SAP_subj_verbs_count,  "overt": non_SAP_subj_count,},
    {"argument type": "non-SAP objects", "total": non_SAP_obj_verbs_count, "overt": non_SAP_obj_count,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

   argument type  total  overt  covert  overt_ratio
    SAP subjects    259     20     239     0.077220
     SAP objects     39      4      35     0.102564
non-SAP subjects    544    157     387     0.288603
 non-SAP objects    300    230      70     0.766667


### Case 3: Obviation

We check counts only on 3-3 verbs.


In [ ]:
from conllu import parse_incr

prox_argument = {"ProxSg", "ProxPl"}
obv_argument = {"ObvSg", "ObvPl"}
prox_subj_marker = {"3SgProxSubj", "3PlProxSubj"}
prox_obj_marker = {"3SgProxObj", "3PlProxObj"}
obv_subj_marker = {"3SgObvSubj", "3PlObvSubj"}
obv_obj_marker = {"3SgObvObj", "3PlObvObj"}

# TODO: Check if inanimate obviatives are tagged with ProxSg/ObvSg etc.
#       I don't think the CGs fully handle these either...

prox_subj_count = prox_obj_count = obv_subj_count = obv_obj_count = 0
third_person_direct_verb_count = third_person_inverse_verb_count = 0

def is_third_person_transitive(tok):
    return is_third_person_direct(tok) or is_third_person_inverse(tok)

# e.g. 3SgProxSubj and 3PlObvObj
def is_third_person_direct(tok):
    xpos = tok.get("xpos")
    if not xpos: return False
    return any(m in xpos for m in prox_subj_marker) and any(m in xpos for m in obv_obj_marker)

# e.g. 3SgObvSubj and 3PlProxObj
def is_third_person_inverse(tok):
    xpos = tok.get("xpos")
    if not xpos: return False
    return any(m in xpos for m in obv_subj_marker) and any(m in xpos for m in prox_obj_marker)

with open(CORPUS_PATH, encoding="utf-8") as f:
    for sent in parse_incr(f):
        for tok in sent:
            xpos = tok.get("xpos")
            if not xpos:
                continue

            # argument counts
            if any(m in xpos for m in prox_argument) and tok["deprel"] == "nsubj" and is_third_person_transitive(sent[tok["head"]-1]):
                prox_subj_count += 1
            if any(m in xpos for m in prox_argument) and tok["deprel"] == "obj" and is_third_person_transitive(sent[tok["head"]-1]):
                prox_obj_count += 1
            if any(m in xpos for m in obv_argument) and tok["deprel"] == "nsubj":
                obv_subj_count += 1
            if any(m in xpos for m in obv_argument) and tok["deprel"] == "obj":
                obv_obj_count += 1

            # total (verb morphology) counts
            if is_third_person_direct(tok):
                third_person_direct_verb_count += 1
            if is_third_person_inverse(tok):
                third_person_inverse_verb_count += 1

print(
    f"Proximate subjects: {prox_subj_count}",
    f"Proximate objects: {prox_obj_count}",
    f"Obviative subjects: {obv_subj_count}",
    f"Obviative objects: {obv_obj_count}",
    sep="\n",
)


# TODO: I get 76 total arguments here, but 74 in the word_order.ipynb... 
#       need to figure out why the discrepancy. probably the logic doesn't match up 100% somewhere

Proximate subjects: 14
Proximate objects: 1
Obviative subjects: 0
Obviative objects: 49


In [38]:
# create stats tables
import pandas as pd

stats = [
    {"argument type": "proximate subject", "total": third_person_direct_verb_count, "overt": prox_subj_count,},
    {"argument type": "obviative object", "total": third_person_direct_verb_count,  "overt": obv_obj_count,},
    {"argument type": "proximate object", "total": third_person_inverse_verb_count,  "overt": prox_obj_count,},
    {"argument type": "obviative subject", "total": third_person_inverse_verb_count, "overt": obv_subj_count,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

    argument type  total  overt  covert  overt_ratio
proximate subject     72     14      58     0.194444
 obviative object     72     49      23     0.680556
 proximate object      1      1       0     1.000000
obviative subject      1      0       1     0.000000
